<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</figure>

# 🔍 SHAP Values — Explicabilidad de Modelos

**Curso:** Ciencia de Datos  
**Tema:** Interpretabilidad · Clase Práctica

---

## ¿Qué problema resuelve SHAP?

Los modelos complejos hacen buenas predicciones pero no nos dicen *por qué*.

**SHAP (SHapley Additive exPlanations)** responde la pregunta:

> *¿Cuánto contribuyó cada variable a la predicción para **esta** observación específica?*

La idea central viene de la **Teoría de Juegos Cooperativos** (Shapley, 1953).

---

## Parte 1 — Intuición: El Reparto Justo

### Analogía: tres vendedores y una venta

Tres vendedores (A, B, C) trabajan juntos y generan distintas ganancias según cómo se combinen:

| Coalición | Ganancia |
|-----------|----------|
| {} vacía  | 0        |
| {A}       | 300      |
| {B}       | 200      |
| {C}       | 100      |
| {A,B}     | 600      |
| {A,C}     | 500      |
| {B,C}     | 400      |
| {A,B,C}   | 900      |

**Pregunta:** ¿Cómo repartir los $900 de manera justa?

La respuesta de Shapley: calcular la **contribución marginal promedio** de cada jugador en **todos los órdenes posibles** de llegada.

In [ ]:
from itertools import permutations
from math import factorial

jugadores = ['A', 'B', 'C']

def ganancia(coalicion: frozenset) -> float:
    tabla = {
        frozenset():              0,
        frozenset(['A']):       300,
        frozenset(['B']):       200,
        frozenset(['C']):       100,
        frozenset(['A','B']):   600,
        frozenset(['A','C']):   500,
        frozenset(['B','C']):   400,
        frozenset(['A','B','C']): 900,
    }
    return tabla[coalicion]

def shapley_value(jugador, jugadores, funcion_valor):
    n = len(jugadores)
    total = 0
    for orden in permutations(jugadores):
        idx   = orden.index(jugador)
        antes = frozenset(orden[:idx])
        con_jugador = antes | frozenset([jugador])
        contribucion = funcion_valor(con_jugador) - funcion_valor(antes)
        total += contribucion
    return total / factorial(n)

print("=" * 40)
print("  Shapley Values — Vendedores")
print("=" * 40)
suma_total = 0
for j in jugadores:
    sv = shapley_value(j, jugadores, ganancia)
    print(f"  Vendedor {j}: ${sv:.2f}")
    suma_total += sv
print("-" * 40)
print(f"  Total repartido: ${suma_total:.2f}  (debe ser $900)")
print("=" * 40)

### 🧮 La fórmula de Shapley

Para cada jugador $i$:

$$\phi_i = \frac{1}{|N|!} \sum_{\text{orden}} \left[ v(S_i \cup \{i\}) - v(S_i) \right]$$

donde $S_i$ es el conjunto de jugadores que llegaron **antes** que $i$ en ese orden.

**Propiedades (axiomas de Shapley):**
1. **Eficiencia:** $\sum_i \phi_i = v(N)$ — se reparte todo
2. **Simetría:** jugadores idénticos reciben el mismo valor
3. **Jugador nulo:** si alguien no aporta nada, recibe 0
4. **Linealidad:** si combinamos dos juegos, los valores se suman

---

## Parte 2 — De Juegos a Machine Learning

| Teoría de Juegos | Machine Learning |
|-----------------|------------------|
| Jugadores | Variables (features) |
| Coalición | Subconjunto de features |
| Valor de coalición | Predicción con ese subconjunto |
| Shapley value $\phi_i$ | SHAP value de cada feature para **una observación** |

El modelo genera una predicción $\hat{y}$. SHAP la descompone:

$$\hat{y} = \underbrace{E[f(x)]}_{\text{baseline}} + \phi_1 + \phi_2 + \cdots + \phi_p$$

- El **baseline** $E[f(x)]$ es la predicción promedio sobre el dataset de entrenamiento
- Cada $\phi_j$ indica cuánto *alejó* o *acercó* la feature $j$ la predicción del baseline

---

## Parte 3 — ¿Cómo se calcula SHAP? Paso a paso

Trabajamos con una casa concreta:

| Feature | Valor observado | Media del train |
|---------|----------------|-----------------|
| `metros` | 50 | ~120 |
| `hab` | 3 | ~3 |
| `garage` | 1 | ~0.5 |

---

### Paso 1 — El modelo predice un valor

**Usamos regresión lineal porque es el modelo que los alumnos comentaron en clase que estaban viendo en otras materias (en particular este modelo puede ser explicado, de forma más simple, sin usar esta técnica)**


$$\hat{y} = \beta_0 + \beta_{\text{metros}} \cdot 50 + \beta_{\text{hab}} \cdot 3 + \beta_{\text{garage}} \cdot 1$$

---

### Paso 2 — Definir el baseline

Si no supiéramos nada de la casa, el modelo predice el promedio de todo el train:

$$E[f(x)] = \text{promedio de predicciones sobre el train set}$$

La pregunta que queremos responder es: **¿por qué la predicción de esta casa es diferente al baseline?**

---

### Paso 3 — Probar todos los órdenes de llegada de las features

Imaginá que las features **llegan de a una** al modelo. Cada vez que llega una, medimos cuánto cambia la predicción.

Con 3 features hay $3! = 6$ órdenes posibles:

```
1. metros → hab → garage
2. metros → garage → hab
3. hab → metros → garage
4. hab → garage → metros
5. garage → metros → hab
6. garage → hab → metros
```

---

### Paso 4 — Calcular la contribución marginal en cada orden

Tomemos el orden `metros → hab → garage` y calculemos el aporte de **`hab`**:

| Momento | Qué sabe el modelo | Predicción |
|---------|-------------------|------------|
| Antes de `hab` | metros=50, hab=**media**, garage=**media** | $f(50, \bar{x}_{\text{hab}}, \bar{x}_{\text{garage}})$ |
| Después de `hab` | metros=50, hab=**3**, garage=**media** | $f(50, 3, \bar{x}_{\text{garage}})$ |
| **Contribución marginal** | diferencia | $f(50, 3, \bar{x}_{\text{garage}}) - f(50, \bar{x}_{\text{hab}}, \bar{x}_{\text{garage}})$ |

> Cuando una feature "todavía no llegó", se reemplaza por su **valor promedio** en el train set.

---

### Paso 5 — Promediar las 6 contribuciones

Se repite el paso 4 para los 6 órdenes. El SHAP value es el **promedio** de las 6 contribuciones marginales:

$$\phi_{\text{hab}} = \frac{1}{6} \sum_{\text{6 órdenes}} \text{contribución marginal de hab en ese orden}$$

---

### Paso 6 — En un modelo lineal esto colapsa a fórmula cerrada

Como el modelo es lineal, todas las contribuciones marginales son iguales sin importar el orden. El promedio sobre los 6 órdenes siempre da:

$$\phi_j = \beta_j \cdot (x_j - \bar{x}_j)$$

Para nuestra casa:

```
φ_metros  = 500   · (50  − 120) = 500   · (−70) = −35,000   ▼
φ_garage  = 20000 · (1   − 0.5) = 20000 · (+0.5) = +10,000  ▲
φ_hab     = 15000 · (3   − 3.0) = 15000 · (0)   =       0  —
```

---

### Paso 7 — Verificar la propiedad de eficiencia

$$E[f(x)] + \phi_{\text{metros}} + \phi_{\text{hab}} + \phi_{\text{garage}} = \hat{y}$$

Esto **siempre se cumple**. Es la propiedad más importante de Shapley: la suma de las contribuciones cierra exactamente con la predicción.

---

### Resumen visual del waterfall

```
E[f(x)]  ──────────────────────────────┐
                                        │
         + φ_garage  (+10,000)  ████ rojo
         + φ_hab     (     0)   (sin barra)
         + φ_metros  (−35,000)  ████████ azul
                                        │
ŷ  ◄────────────────────────────────────┘
```

El waterfall plot es exactamente este camino: barra por barra desde el baseline hasta $\hat{y}$.  
El **número dentro de cada barra** es el valor de $\phi_j$ — cuántos pesos sube o baja la predicción esa feature.

---

## Parte 4 — Implementación: Precio de Casas

**Features:**
- `metros` — superficie en m²
- `hab` — cantidad de habitaciones
- `garage` — 1 si tiene garage, 0 si no

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
n = 100

metros = np.random.uniform(40, 200, n)
hab    = np.random.randint(1, 6, n).astype(float)
garage = np.random.randint(0, 2, n).astype(float)
precio = 500*metros + 15000*hab + 20000*garage + np.random.normal(0, 8000, n)

df = pd.DataFrame({'metros': metros, 'hab': hab, 'garage': garage, 'precio': precio})

print("Primeras 5 filas:")
print(df.head().round(1).to_string())
print(f"\nPrecio promedio: ${df.precio.mean():,.0f}")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X = df[['metros', 'hab', 'garage']]
y = df['precio']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LinearRegression()
modelo.fit(X_train, y_train)

print("=" * 45)
print("  Coeficientes del modelo lineal")
print("=" * 45)
print(f"  Intercepto : ${modelo.intercept_:,.0f}")
for feat, coef in zip(X.columns, modelo.coef_):
    print(f"  {feat:10s} : ${coef:,.0f} por unidad")
print("-" * 45)
print(f"  R² en test : {modelo.score(X_test, y_test):.4f}")
print("=" * 45)

### Cálculo manual: $\phi_j = \beta_j \cdot (x_j - \bar{x}_j)$

In [ ]:
# Observación a explicar: primera del test set
obs      = X_test.iloc[[0]]
pred     = modelo.predict(obs)[0]
baseline = modelo.predict(X_train).mean()

print("Observación a explicar:")
print(obs.to_string())
print(f"\nPredicción del modelo : ${pred:,.0f}")
print(f"Baseline  E[f(x)]     : ${baseline:,.0f}")
print(f"Diferencia a explicar : ${pred - baseline:+,.0f}")

print("\n" + "=" * 58)
print("  SHAP Values — cálculo manual:  φⱼ = βⱼ · (xⱼ − x̄ⱼ)")
print("=" * 58)

medias = X_train.mean()
shap_manual = {}
for feat, coef in zip(X.columns, modelo.coef_):
    xj  = obs[feat].values[0]
    mu  = medias[feat]
    phi = coef * (xj - mu)
    shap_manual[feat] = phi
    direccion = "▲ sube" if phi > 0 else ("▼ baja" if phi < 0 else "— neutro")
    print(f"  {feat:8s}: β={coef:>8,.0f}  x={xj:>7.1f}  μ={mu:>7.1f}  φ={phi:>+10,.0f}  {direccion}")

print("-" * 58)
suma_phi = sum(shap_manual.values())
print(f"  Σ(φ)                  = ${suma_phi:+,.0f}")
print(f"  Baseline + Σ(φ)       = ${baseline + suma_phi:,.0f}")
print(f"  Predicción del modelo = ${pred:,.0f}  ✓")
print("=" * 58)

---

## Parte 5 — Verificación con la librería `shap`

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)

try:
    import shap
    from packaging.version import Version
    if Version(shap.__version__) < Version("0.46"):
        raise ImportError("versión desactualizada")
except ImportError:
    print("Instalando shap>=0.46 ...")
    install("shap>=0.46")
    import shap

print(f"shap versión: {shap.__version__}")

In [ ]:
import shap

# LinearExplainer: exacto para regresiones lineales
explainer   = shap.LinearExplainer(modelo, X_train, feature_perturbation='interventional')
shap_values = explainer(X_test)

print("Comparación: cálculo manual vs librería shap")
print("-" * 50)
for feat in X.columns:
    idx = list(X.columns).index(feat)
    lib = shap_values.values[0][idx]
    man = shap_manual[feat]
    print(f"  {feat:8s}: manual={man:>+10,.0f}   librería={lib:>+10,.0f}   Δ={abs(lib-man):.1f}")

---

## Parte 6 — Visualizaciones

### 6.1 Waterfall Plot — Una observación

Representa el camino desde $E[f(x)]$ hasta $\hat{y}$, barra por barra.

- **Eje horizontal:** valor de referencia ($E[f(x)]$) y predicción final ($\hat{y}$)
- **Barras rojas:** $\phi_j > 0$ — la feature sube la predicción respecto al baseline
- **Barras azules:** $\phi_j < 0$ — la feature baja la predicción respecto al baseline
- **Número dentro de cada barra:** el valor de $\phi_j$ — cuántos pesos mueve la predicción

In [ ]:
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

shap.plots.waterfall(shap_values[0], show=True)

### 6.2 Force Plot — Vista compacta

Misma información que el waterfall pero en horizontal. Útil para comparar varias observaciones a la vez.

In [ ]:
shap.plots.force(
    shap_values.base_values[0],
    shap_values.values[0],
    X_test.iloc[0],
    matplotlib=True
)

### 6.3 Beeswarm — Todo el test set

Muestra la distribución de SHAP values para **todas las observaciones** a la vez.

- Cada punto = una observación
- **Eje X:** valor de $\phi_j$ (derecha = sube predicción, izquierda = baja)
- **Color:** valor de la feature (rojo = alto, azul = bajo)
- **Orden vertical:** importancia global = promedio de $|\phi_j|$

💡 *En el modelo lineal el patrón es perfectamente monótono: valores altos de `metros` siempre tienen $\phi > 0$.*

In [ ]:
shap.plots.beeswarm(shap_values, show=True)

### 6.4 Bar Plot — Importancia global

Resume la importancia de cada feature como $\text{mean}(|\phi_j|)$ sobre todo el dataset.

In [ ]:
shap.plots.bar(shap_values, show=True)

---

## Resumen

| Concepto | Descripción |
|----------|-------------|
| **SHAP value** $\phi_j$ | Contribución de la feature $j$ a la predicción de **una** observación |
| **Baseline** $E[f(x)]$ | Predicción promedio del modelo sobre el train set |
| **Eficiencia** | $\hat{y} = E[f(x)] + \sum_j \phi_j$ siempre se cumple |
| **Cálculo** | Promedio de contribuciones marginales sobre todos los órdenes |
| **Lineal (fórmula cerrada)** | $\phi_j = \beta_j \cdot (x_j - \bar{x}_j)$ |
| **Número en la barra** | Valor de $\phi_j$: cuánto sube o baja la predicción esa feature |
| **Waterfall** | Camino desde el baseline hasta $\hat{y}$, barra por barra |
| **Beeswarm** | Distribución de $\phi_j$ para todas las observaciones |

---

## 🏋️ Ejercicios Propuestos

1. Calcular manualmente los Shapley values del juego de vendedores verificando **cada orden** para el jugador B.

2. Agregar una 4ª feature `antigüedad` al dataset. ¿Cómo cambia el beeswarm? ¿Qué feature queda con menor SHAP global?

3. Elegir 3 observaciones del test set con predicciones muy distintas (baja, media, alta). Comparar sus waterfall plots. ¿Qué feature domina en cada caso?

4. Verificar la propiedad de eficiencia para 5 observaciones al azar: comprobar que `baseline + Σφ = ŷ`.

---

## 📚 Referencias

- Lundberg & Lee (2017). *A Unified Approach to Interpreting Model Predictions*. NeurIPS.
- Shapley, L.S. (1953). *A value for n-person games*. Contributions to the Theory of Games.
- Molnar, C. (2022). *Interpretable Machine Learning*. https://christophm.github.io/interpretable-ml-book/